# Chapter 2 (Burden & Burden): Solving Nonlinear Equations in One Variable

This notebook is **self-contained** and focuses on the core Chapter 2 material: root-finding methods, convergence ideas, worked demonstrations, and suggested exercises.

It was prepared using publicly available online references for method summaries (not local project files).

## Topic Map (Chapter 2)

1. Problem setup and stopping criteria
2. Bisection method
3. False position (Regula Falsi)
4. Fixed-point iteration
5. Newton's method
6. Secant method
7. Error behavior and order of convergence
8. Convergence acceleration (Aitken and Steffensen)
9. Muller's method (quadratic interpolation)

We will use a few test equations repeatedly to compare methods.

In [6]:
import math
import cmath
from typing import Callable, List, Dict, Tuple

def print_table(rows: List[Dict], columns: List[str], max_rows: int = 12) -> None:
    show = rows[:max_rows]
    if not show:
        print('(no rows)')
        return
    header = ' | '.join(columns)
    print(header)
    print('-' * len(header))
    for r in show:
        vals = []
        for c in columns:
            v = r.get(c, '')
            if isinstance(v, float):
                vals.append(f'{v:.12g}')
            else:
                vals.append(str(v))
        print(' | '.join(vals))
    if len(rows) > max_rows:
        print(f'... ({len(rows)-max_rows} more rows)')

def bisection(f: Callable[[float], float], a: float, b: float, tol: float = 1e-10, max_iter: int = 100) -> Tuple[float, List[Dict]]:
    fa, fb = f(a), f(b)
    if fa == 0:
        return a, [{'n': 0, 'a': a, 'b': b, 'p': a, 'f(p)': fa, 'width': b-a}]
    if fb == 0:
        return b, [{'n': 0, 'a': a, 'b': b, 'p': b, 'f(p)': fb, 'width': b-a}]
    if fa * fb > 0:
        raise ValueError('Bisection requires f(a) and f(b) of opposite sign.')

    hist = []
    for n in range(1, max_iter + 1):
        p = a + (b - a) / 2
        fp = f(p)
        hist.append({'n': n, 'a': a, 'b': b, 'p': p, 'f(p)': fp, 'width': b-a})

        if abs(fp) < tol or abs(b - a) < tol:
            return p, hist

        if fa * fp < 0:
            b, fb = p, fp
        else:
            a, fa = p, fp

    return p, hist

def false_position(f: Callable[[float], float], a: float, b: float, tol: float = 1e-10, max_iter: int = 100) -> Tuple[float, List[Dict]]:
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError('False position requires f(a) and f(b) of opposite sign.')

    hist = []
    c = a
    for n in range(1, max_iter + 1):
        c = (a * fb - b * fa) / (fb - fa)
        fc = f(c)
        hist.append({'n': n, 'a': a, 'b': b, 'c': c, 'f(c)': fc, 'width': b-a})

        if abs(fc) < tol or abs(b - a) < tol:
            return c, hist

        if fa * fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc

    return c, hist

def fixed_point(g: Callable[[float], float], x0: float, tol: float = 1e-10, max_iter: int = 100) -> Tuple[float, List[Dict]]:
    hist = []
    x = x0
    for n in range(1, max_iter + 1):
        xn = g(x)
        hist.append({'n': n, 'x_n': x, 'x_n+1': xn, '|dx|': abs(xn-x)})
        if abs(xn - x) < tol:
            return xn, hist
        x = xn
    return x, hist

def newton(f: Callable[[float], float], df: Callable[[float], float], x0: float, tol: float = 1e-10, max_iter: int = 50) -> Tuple[float, List[Dict]]:
    hist = []
    x = x0
    for n in range(1, max_iter + 1):
        fx = f(x)
        dfx = df(x)
        if dfx == 0:
            raise ZeroDivisionError('Derivative is zero during Newton iteration.')
        xn = x - fx / dfx
        hist.append({'n': n, 'x_n': x, 'f(x_n)': fx, '|dx|': abs(xn-x)})
        if abs(xn - x) < tol:
            return xn, hist
        x = xn
    return x, hist

def secant(f: Callable[[float], float], x0: float, x1: float, tol: float = 1e-10, max_iter: int = 80) -> Tuple[float, List[Dict]]:
    hist = []
    for n in range(1, max_iter + 1):
        f0, f1 = f(x0), f(x1)
        den = (f1 - f0)
        if den == 0:
            raise ZeroDivisionError('Secant denominator became zero.')
        x2 = x1 - f1 * (x1 - x0) / den
        hist.append({'n': n, 'x_n-1': x0, 'x_n': x1, 'x_n+1': x2, '|dx|': abs(x2-x1)})
        if abs(x2 - x1) < tol:
            return x2, hist
        x0, x1 = x1, x2
    return x1, hist

def aitken_delta2(seq: List[float]) -> List[float]:
    out = []
    for n in range(len(seq) - 2):
        x0, x1, x2 = seq[n], seq[n+1], seq[n+2]
        den = x2 - 2*x1 + x0
        if den == 0:
            out.append(float('nan'))
        else:
            out.append(x0 - (x1 - x0)**2 / den)
    return out

def steffensen(f: Callable[[float], float], x0: float, tol: float = 1e-10, max_iter: int = 50) -> Tuple[float, List[Dict]]:
    hist = []
    x = x0
    for n in range(1, max_iter + 1):
        fx = f(x)
        gx_num = f(x + fx) - fx
        gx_den = fx
        if gx_den == 0:
            return x, hist
        g = gx_num / gx_den
        if g == 0:
            raise ZeroDivisionError('Steffensen divided difference became zero.')
        xn = x - fx / g
        hist.append({'n': n, 'x_n': x, 'f(x_n)': fx, '|dx|': abs(xn - x)})
        if abs(xn - x) < tol:
            return xn, hist
        x = xn
    return x, hist

def muller(f: Callable[[complex], complex], x0: complex, x1: complex, x2: complex, tol: float = 1e-10, max_iter: int = 50) -> Tuple[complex, List[Dict]]:
    hist = []
    for n in range(1, max_iter + 1):
        h0 = x1 - x0
        h1 = x2 - x1
        d0 = (f(x1) - f(x0)) / h0
        d1 = (f(x2) - f(x1)) / h1
        a = (d1 - d0) / (h1 + h0)
        b = a*h1 + d1
        c = f(x2)

        rad = cmath.sqrt(b*b - 4*a*c)
        den1 = b + rad
        den2 = b - rad
        den = den1 if abs(den1) > abs(den2) else den2
        if den == 0:
            raise ZeroDivisionError('Muller denominator became zero.')

        x3 = x2 - (2*c) / den
        hist.append({'n': n, 'x_n': x2, 'f(x_n)': f(x2), '|dx|': abs(x3-x2)})
        if abs(x3 - x2) < tol:
            return x3, hist
        x0, x1, x2 = x1, x2, x3
    return x2, hist

def estimate_order(errors: List[float]) -> float:
    # Uses p ≈ ln(e_{n+1}/e_n) / ln(e_n/e_{n-1}) on the tail of the error sequence.
    vals = []
    for i in range(2, len(errors)):
        e_nm1, e_n, e_np1 = errors[i-2], errors[i-1], errors[i]
        if e_nm1 > 0 and e_n > 0 and e_np1 > 0 and e_n != e_nm1:
            num = math.log(e_np1 / e_n)
            den = math.log(e_n / e_nm1)
            if den != 0:
                vals.append(num / den)
    return vals[-1] if vals else float('nan')

print('All method helpers loaded.')

All method helpers loaded.


## 1) Problem Setup and Stopping Criteria

For a nonlinear equation, we rewrite as `f(x)=0` and seek a root `p`.

Common stopping checks:
- Residual small: `|f(x_n)| < tol`
- Step small: `|x_n - x_{n-1}| < tol`
- Relative step small: `|x_n - x_{n-1}| / |x_n| < tol`
- Max iteration reached

In practice we use at least one step-based check plus a residual check.

## 2) Bisection Method

Idea: Start with `[a,b]` such that `f(a)f(b)<0`, then repeatedly halve interval.

Strengths: guaranteed convergence for continuous `f` with sign change.
Weakness: linear convergence (slow but robust).

In [7]:
# Example: x^3 + 4x^2 - 10 = 0 on [1, 2]
f = lambda x: x**3 + 4*x**2 - 10
root_bi, hist_bi = bisection(f, 1.0, 2.0, tol=1e-12, max_iter=100)
print('Bisection root:', root_bi)
print('f(root):', f(root_bi))
print('iterations:', len(hist_bi))
print_table(hist_bi, ['n', 'a', 'b', 'p', 'f(p)', 'width'], max_rows=8)

Bisection root: 1.3652300134140205
f(root): -1.2612133559741778e-12
iterations: 41
n | a | b | p | f(p) | width
----------------------------
1 | 1 | 2 | 1.5 | 2.375 | 1
2 | 1 | 1.5 | 1.25 | -1.796875 | 0.5
3 | 1.25 | 1.5 | 1.375 | 0.162109375 | 0.25
4 | 1.25 | 1.375 | 1.3125 | -0.848388671875 | 0.125
5 | 1.3125 | 1.375 | 1.34375 | -0.350982666016 | 0.0625
6 | 1.34375 | 1.375 | 1.359375 | -0.0964088439941 | 0.03125
7 | 1.359375 | 1.375 | 1.3671875 | 0.0323557853699 | 0.015625
8 | 1.359375 | 1.3671875 | 1.36328125 | -0.0321499705315 | 0.0078125
... (33 more rows)


### Suggested Exercises (Bisection)
1. For `f(x)=x^5-3x+1`, find an interval that brackets a root and compute to `1e-6`.
2. Compare required iterations from theory `n >= log2((b-a)/eps)` versus actual run.
3. Construct a continuous function with two roots in `[a,b]` and discuss which root bisection converges to under different brackets.

## 3) False Position (Regula Falsi)

Idea: Keep a bracketing interval but replace midpoint by secant x-intercept.

Can converge faster than bisection, but sometimes one endpoint stalls.

In [8]:
root_fp, hist_fp = false_position(f, 1.0, 2.0, tol=1e-12, max_iter=100)
print('False-position root:', root_fp)
print('f(root):', f(root_fp))
print('iterations:', len(hist_fp))
print_table(hist_fp, ['n', 'a', 'b', 'c', 'f(c)', 'width'], max_rows=8)

False-position root: 1.3652300134140698
f(root): -4.476419235288631e-13
iterations: 22
n | a | b | c | f(c) | width
----------------------------
1 | 1 | 2 | 1.26315789474 | -1.60227438402 | 1
2 | 1.26315789474 | 2 | 1.33882783883 | -0.430364748005 | 0.736842105263
3 | 1.33882783883 | 2 | 1.35854634182 | -0.110008788474 | 0.661172161172
4 | 1.35854634182 | 2 | 1.36354744004 | -0.0277620910011 | 0.641453658175
5 | 1.36354744004 | 2 | 1.36480703183 | -0.00698341540117 | 0.636452559958
6 | 1.36480703183 | 2 | 1.36512371788 | -0.00175520903234 | 0.635192968173
7 | 1.36512371788 | 2 | 1.36520330366 | -0.000441063010154 | 0.634876282116
8 | 1.36520330366 | 2 | 1.36522330199 | -0.000110828133424 | 0.634796696337
... (14 more rows)


### Suggested Exercises (False Position)
1. Compare iteration counts of bisection and false position on the same equation and tolerance.
2. Find an example where false position is slow because one endpoint is nearly fixed.
3. Implement Illinois modification and test whether it reduces stalling.

## 4) Fixed-Point Iteration

Rewrite equation as `x=g(x)` and iterate `x_{n+1}=g(x_n)`.

Local rule of thumb: if `|g'(p)|<1` near fixed point `p`, iteration is locally convergent.

In [9]:
# Solve x^3 + x - 1 = 0 via x = 1/(1+x^2)
g = lambda x: 1.0/(1.0 + x*x)
root_fx, hist_fx = fixed_point(g, x0=0.5, tol=1e-12, max_iter=200)
f2 = lambda x: x**3 + x - 1
print('Fixed-point root:', root_fx)
print('f(root):', f2(root_fx))
print('iterations:', len(hist_fx))
print_table(hist_fx, ['n', 'x_n', 'x_n+1', '|dx|'], max_rows=8)

Fixed-point root: 0.6823278038277458
f(root): -6.554756737386924e-13
iterations: 60
n | x_n | x_n+1 | |dx|
----------------------
1 | 0.5 | 0.8 | 0.3
2 | 0.8 | 0.609756097561 | 0.190243902439
3 | 0.609756097561 | 0.728967909801 | 0.11921181224
4 | 0.728967909801 | 0.652999724808 | 0.0759681849928
5 | 0.652999724808 | 0.701061372974 | 0.0480616481661
6 | 0.701061372974 | 0.670471795841 | 0.0305895771324
7 | 0.670471795841 | 0.689877632249 | 0.0194058364078
8 | 0.689877632249 | 0.677538380912 | 0.012339251337
... (52 more rows)


### Suggested Exercises (Fixed Point)
1. For `x^3 + x - 1 = 0`, derive two different `g(x)` forms and test which converges from `x0=0.5`.
2. Numerically estimate `|g'(x_n)|` near the limit and relate it to observed speed.
3. Build a divergent fixed-point map for the same root and explain failure.

## 5) Newton's Method

Update rule:
`x_{n+1} = x_n - f(x_n)/f'(x_n)`

When close to a simple root and assumptions hold, convergence is typically quadratic.

In [10]:
f3 = lambda x: math.cos(x) - x
df3 = lambda x: -math.sin(x) - 1
root_new, hist_new = newton(f3, df3, x0=0.5, tol=1e-14, max_iter=30)
print('Newton root:', root_new)
print('f(root):', f3(root_new))
print('iterations:', len(hist_new))
print_table(hist_new, ['n', 'x_n', 'f(x_n)', '|dx|'], max_rows=8)

Newton root: 0.7390851332151607
f(root): 0.0
iterations: 5
n | x_n | f(x_n) | |dx|
-----------------------
1 | 0.5 | 0.37758256189 | 0.255222417106
2 | 0.755222417106 | -0.0271033118575 | 0.0160807509558
3 | 0.73914166615 | -9.46153806177e-05 | 5.65322290724e-05
4 | 0.739085133921 | -1.18097787105e-09 | 7.0564609711e-10
5 | 0.739085133215 | 0 | 0


### Multiple Root Note (Modified Newton)
For multiplicity `m`, a common modification is:
`x_{n+1} = x_n - m f(x_n)/f'(x_n)`
to recover faster convergence compared with plain Newton.

In [11]:
# Compare plain vs modified Newton for f(x)=(x-1)^2 (double root at x=1)
fm = lambda x: (x-1)**2
dfm = lambda x: 2*(x-1)

r_plain, h_plain = newton(fm, dfm, x0=1.8, tol=1e-14, max_iter=30)

def newton_modified_multiplicity(f, df, x0, m, tol=1e-14, max_iter=30):
    x = x0
    hist = []
    for n in range(1, max_iter+1):
        fx = f(x)
        dfx = df(x)
        xn = x - m * fx / dfx
        hist.append({'n': n, 'x_n': x, '|dx|': abs(xn-x)})
        if abs(xn-x) < tol:
            return xn, hist
        x = xn
    return x, hist

r_mod, h_mod = newton_modified_multiplicity(fm, dfm, x0=1.8, m=2, tol=1e-14, max_iter=30)
print('Plain Newton iterations:', len(h_plain), 'root:', r_plain)
print('Modified Newton iterations:', len(h_mod), 'root:', r_mod)
print_table(h_plain, ['n', 'x_n', '|dx|'], max_rows=6)
print('---')
print_table(h_mod, ['n', 'x_n', '|dx|'], max_rows=6)

Plain Newton iterations: 30 root: 1.000000000745058
Modified Newton iterations: 2 root: 1.0
n | x_n | |dx|
--------------
1 | 1.8 | 0.4
2 | 1.4 | 0.2
3 | 1.2 | 0.1
4 | 1.1 | 0.05
5 | 1.05 | 0.025
6 | 1.025 | 0.0125
... (24 more rows)
---
n | x_n | |dx|
--------------
1 | 1.8 | 0.8
2 | 1 | 1.11022302463e-16


### Suggested Exercises (Newton)
1. Solve `e^{-x} - x = 0` from three different initial guesses and compare behavior.
2. Build a case where Newton diverges or oscillates (bad initial guess).
3. For a known multiple root, compare plain and modified Newton quantitatively.

## 6) Secant Method

Update uses two previous iterates and finite-difference slope (no derivative needed).

Convergence order is superlinear (about `1.618`) for simple roots under standard assumptions.

In [12]:
root_sec, hist_sec = secant(f3, x0=0.0, x1=1.0, tol=1e-14, max_iter=40)
print('Secant root:', root_sec)
print('f(root):', f3(root_sec))
print('iterations:', len(hist_sec))
print_table(hist_sec, ['n', 'x_n-1', 'x_n', 'x_n+1', '|dx|'], max_rows=8)

Secant root: 0.7390851332151607
f(root): 0.0
iterations: 7
n | x_n-1 | x_n | x_n+1 | |dx|
------------------------------
1 | 0 | 1 | 0.685073357326 | 0.314926642674
2 | 1 | 0.685073357326 | 0.736298997614 | 0.0512256402876
3 | 0.685073357326 | 0.736298997614 | 0.739119361912 | 0.00282036429798
4 | 0.736298997614 | 0.739119361912 | 0.739085112127 | 3.42497841654e-05
5 | 0.739119361912 | 0.739085112127 | 0.739085133215 | 2.10875373829e-08
6 | 0.739085112127 | 0.739085133215 | 0.739085133215 | 1.59428026336e-13
7 | 0.739085133215 | 0.739085133215 | 0.739085133215 | 0


### Suggested Exercises (Secant)
1. Use secant on `x^3-2x-5=0` with two different starting pairs.
2. Compare cost vs Newton when derivative is expensive or unavailable.
3. Detect and handle the denominator-near-zero failure case robustly.

## 7) Empirical Convergence Order Comparison

For `f(x)=cos(x)-x`, we estimate convergence order from observed errors against a high-accuracy reference root.

In [18]:
# Recompute all compared methods on the same equation f3(x)=cos(x)-x

true_root = root_new  # from Newton on f3



root_bi_f3, hist_bi_f3 = bisection(f3, 0.0, 1.0, tol=1e-14, max_iter=100)

root_fp_f3, hist_fp_f3 = false_position(f3, 0.0, 1.0, tol=1e-14, max_iter=100)



errs_new = [abs(row['x_n'] - true_root) for row in hist_new if abs(row['x_n'] - true_root) > 0]

errs_sec = [abs(row['x_n'] - true_root) for row in hist_sec if abs(row['x_n'] - true_root) > 0]

errs_fp = [abs(row['c'] - true_root) for row in hist_fp_f3 if abs(row['c'] - true_root) > 0]



# Order estimate (useful for superlinear methods)

print('Estimated order p (Newton):', estimate_order(errs_new))

print('Estimated order p (Secant):', estimate_order(errs_sec))



# Bisection halves interval width exactly each step (robust linear indicator)

w = [row['width'] for row in hist_bi_f3]

print('Width ratio w_{n+1}/w_n (Bisection):', w[5]/w[4])



# False-position linear behavior: use non-tiny residual-error ratios to avoid roundoff noise

fp_ratios = []

for i in range(len(errs_fp)-1):

    if errs_fp[i] > 1e-10 and errs_fp[i+1] > 1e-10:

        fp_ratios.append(errs_fp[i+1]/errs_fp[i])

print('Representative error ratio e_{n+1}/e_n (False Position):', fp_ratios[-1] if fp_ratios else float('nan'))

Estimated order p (Newton): 1.9970095121466291
Estimated order p (Secant): 1.595292287070036
Width ratio w_{n+1}/w_n (Bisection): 0.5
Representative error ratio e_{n+1}/e_n (False Position): 0.050092561433828175


## 8) Aitken's `Δ²` and Steffensen's Method

Aitken accelerates linearly convergent sequences.
Steffensen applies this idea directly to fixed-point style updates and often achieves quadratic behavior without explicit derivatives.

In [14]:
# Build a linear-convergent sequence from fixed-point iteration and accelerate it with Aitken
seq = [0.5]
for _ in range(8):
    seq.append(g(seq[-1]))
ait = aitken_delta2(seq)

print('Original sequence (first 7):')
print([round(v, 12) for v in seq[:7]])
print('Aitken accelerated (first 5):')
print([round(v, 12) if not math.isnan(v) else None for v in ait[:5]])

root_st, hist_st = steffensen(f2, x0=0.5, tol=1e-12, max_iter=30)
print('Steffensen root for x^3+x-1=0:', root_st)
print('f(root):', f2(root_st))
print('iterations:', len(hist_st))
print_table(hist_st, ['n', 'x_n', 'f(x_n)', '|dx|'], max_rows=8)

Original sequence (first 7):
[0.5, 0.8, 0.609756097561, 0.728967909801, 0.652999724808, 0.701061372974, 0.670471795841]
Aitken accelerated (first 5):
[0.683582089552, 0.683043871228, 0.682568149918, 0.68243745028, 0.682368905207]
Steffensen root for x^3+x-1=0: 0.6823278038280193
f(root): -1.1102230246251565e-16
iterations: 7
n | x_n | f(x_n) | |dx|
-----------------------
1 | 0.5 | -0.375 | 0.282352941176
2 | 0.782352941176 | 0.261212497456 | 0.0742599630874
3 | 0.708092978089 | 0.0631277281581 | 0.0238914479764
4 | 0.684201530113 | 0.00449797867873 | 0.00186356033256
5 | 0.68233796978 | 2.43650881322e-05 | 1.01656522695e-05
6 | 0.682327804128 | 7.1856298689e-10 | 2.99811730997e-10
7 | 0.682327803828 | 2.22044604925e-16 | 1.11022302463e-16


### Suggested Exercises (Aitken and Steffensen)
1. Apply Aitken to a sequence with known linear convergence factor and verify acceleration.
2. Compare fixed-point vs Steffensen on the same `g(x)` and `x0`.
3. Investigate numerical instability when `Δ² x_n` is very small.

## 9) Muller's Method

Muller fits a quadratic through three points and takes a root of that quadratic as the next iterate.

It can produce complex iterates even from real starts, which is useful for finding complex roots.

In [15]:
# Example with complex roots: x^3 - 1 = 0
fmul = lambda z: z**3 - 1

root_mul, hist_mul = muller(fmul, x0=0.0, x1=0.5+0.5j, x2=1.0j, tol=1e-12, max_iter=40)
print('Muller root approximation:', root_mul)
print('f(root):', fmul(root_mul))
print('iterations:', len(hist_mul))

# Show the last few steps only for compactness
for row in hist_mul[-5:]:
    print(row['n'], row['x_n'], row['|dx|'])

Muller root approximation: (-0.5+0.8660254037844386j)
f(root): (-2.220446049250313e-16+1.1102230246251565e-16j)
iterations: 7
3 (-0.5331044295259608+0.8461877528615964j) 0.03967748438931741
4 (-0.4995817813318364+0.867413558512207j) 0.001447643388571704
5 (-0.5000025294038372+0.8660284078199056j) 3.927152342113948e-06
6 (-0.49999999992712285+0.866025403777118j) 7.324391752269746e-11
7 (-0.5+0.8660254037844387j) 1.1102230246251565e-16


### Suggested Exercises (Muller)
1. Use Muller on `x^3-1` with different initial triples and observe which root is selected.
2. Compare secant vs Muller on a real-root problem and count iterations.
3. Investigate what happens when the quadratic discriminant is near zero.

## Method Selection Guide

- Use **Bisection** when guaranteed robustness is priority.
- Use **False Position** when you want bracketing plus often faster progress.
- Use **Newton** when derivative is available and initial guess is good.
- Use **Secant** when derivative is unavailable but fast local convergence is desired.
- Use **Steffensen** to accelerate fixed-point methods without explicit derivative.
- Use **Muller** when complex roots are acceptable or desired.

## Net References Used

Public web pages consulted for method summaries and convergence facts:
- https://en.wikipedia.org/wiki/Bisection_method
- https://en.wikipedia.org/wiki/Regula_falsi
- https://en.wikipedia.org/wiki/Fixed-point_iteration
- https://en.wikipedia.org/wiki/Newton%27s_method
- https://en.wikipedia.org/wiki/Secant_method
- https://en.wikipedia.org/wiki/Aitken%27s_delta-squared_process
- https://en.wikipedia.org/wiki/Steffensen%27s_method
- https://en.wikipedia.org/wiki/Muller%27s_method
- https://yaningliucudenver.github.io/Numerical-Analysis-I/bookchapter2-2.html
- https://faculty.washington.edu/trogdon/105A/html/Lecture3.html

This notebook intentionally uses original explanations and code implementations.